# Abstractive Text Summarization with DistilBART

A concise, reproducible notebook for Project 01. It uses the modular repository code rather than duplicating production logic.

> Responsible use: review every summary and do not process confidential text in public environments.

## 1. Setup

In [ ]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.data_preprocessing import prepare_dataframe
from src.summarization_model import GenerationSettings, TransformerSummarizer
from src.model_evaluation import evaluate_dataframe, save_evaluation_outputs

## 2. Load Safe Sample Data

For benchmark work, switch to `src.dataset_loader.load_public_dataset` with a bounded XSum or CNN/DailyMail subset.

In [ ]:
data_path = PROJECT_ROOT / 'data' / 'sample_summaries.csv'
raw_df = pd.read_csv(data_path)
df = prepare_dataframe(raw_df)
df[['id', 'title', 'article_words', 'summary_words']].head()

## 3. Inspect Lengths and Preprocessing

In [ ]:
df[['article_words', 'summary_words']].describe()

## 4. Configure the Transformer

The first inference downloads `sshleifer/distilbart-cnn-12-6`. CPU execution may take time.

In [ ]:
settings = GenerationSettings(
    min_length=30,
    max_length=120,
    num_beams=4,
    length_penalty=2.0,
    no_repeat_ngram_size=3,
    early_stopping=True,
)
summarizer = TransformerSummarizer()
settings

## 5. Generate One Summary

In [ ]:
result = summarizer.summarize(df.loc[0, 'article'], settings)
print(result.summary)
result.to_dict()

## 6. Evaluate ROUGE, BERTScore and Latency

Set `compute_bert_score=False` for a faster smoke test. Publish only metrics produced by an actual run.

In [ ]:
results, metrics = evaluate_dataframe(
    df, summarizer, settings, compute_bert_score=True
)
metrics

In [ ]:
results[['id', 'reference_summary', 'transformer_summary', 'inference_seconds', 'compression_ratio']]

## 7. Save Reproducible Outputs

In [ ]:
output_dir = PROJECT_ROOT / 'outputs' / 'runs' / 'notebook_run'
save_evaluation_outputs(results, metrics, output_dir)
print(output_dir)

## 8. Error Analysis

Review missing facts, entity/number changes, hallucination, repetition, over-compression, and domain mismatch. Record actual examples in `outputs/error_analysis_examples.md`.